# Week 3, Notebook 2: Generative AI with PyTorch
## VAE on MNIST — Generate Handwritten Digits

**What you'll build:** A VAE that generates new handwritten digits.

**Time estimate:** 45–60 minutes

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: Load MNIST (built from scratch — no torchvision needed)

In [ ]:
# ============================================================
# Generate a simple handwritten-digit-like dataset
# (Works without torchvision — self-contained!)
# ============================================================
import struct
import os

def generate_digit_dataset(n_per_digit=200):
    """Generate synthetic digit-like images (8x8) for demonstration.
    
    If you have torchvision, replace this with MNIST for better results.
    """
    images = []
    labels = []
    
    for digit in range(10):
        for _ in range(n_per_digit):
            img = np.random.rand(8, 8) * 0.1  # background noise
            
            # Draw digit-like patterns
            noise = np.random.randn() * 0.3
            if digit == 0:
                img[1:7, 1] = 0.8 + noise * 0.1; img[1:7, 6] = 0.8 + noise * 0.1
                img[1, 1:7] = 0.8 + noise * 0.1; img[6, 1:7] = 0.8 + noise * 0.1
            elif digit == 1:
                img[1:7, 4] = 0.9 + noise * 0.1
            elif digit == 2:
                img[1, 1:7] = 0.8; img[3, 1:7] = 0.8; img[6, 1:7] = 0.8
                img[1:3, 6] = 0.8; img[4:7, 1] = 0.8
            elif digit == 3:
                img[1, 1:7] = 0.8; img[3, 1:7] = 0.8; img[6, 1:7] = 0.8
                img[1:7, 6] = 0.8
            elif digit == 4:
                img[1:4, 1] = 0.8; img[3, 1:7] = 0.8; img[1:7, 6] = 0.8
            elif digit == 5:
                img[1, 1:7] = 0.8; img[3, 1:7] = 0.8; img[6, 1:7] = 0.8
                img[1:3, 1] = 0.8; img[4:7, 6] = 0.8
            elif digit == 6:
                img[1:7, 1] = 0.8; img[1, 1:7] = 0.8; img[3, 1:7] = 0.8
                img[6, 1:7] = 0.8; img[4:7, 6] = 0.8
            elif digit == 7:
                img[1, 1:7] = 0.8; img[1:7, 6] = 0.8
            elif digit == 8:
                img[1:7, 1] = 0.8; img[1:7, 6] = 0.8
                img[1, 1:7] = 0.8; img[3, 1:7] = 0.8; img[6, 1:7] = 0.8
            elif digit == 9:
                img[1:4, 1] = 0.8; img[1:7, 6] = 0.8
                img[1, 1:7] = 0.8; img[3, 1:7] = 0.8
            
            # Add random noise and slight shifts
            img += np.random.randn(8, 8) * 0.08
            shift_r = np.random.randint(-1, 2)
            shift_c = np.random.randint(-1, 2)
            img = np.roll(np.roll(img, shift_r, axis=0), shift_c, axis=1)
            img = np.clip(img, 0, 1)
            
            images.append(img)
            labels.append(digit)
    
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels)
    idx = np.random.permutation(len(images))
    return images[idx], labels[idx]


X_images, y_digits = generate_digit_dataset(300)
print(f"Dataset: {X_images.shape} images, {len(np.unique(y_digits))} digits")

# Show samples
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for digit in range(10):
    mask = y_digits == digit
    sample = X_images[mask][0]
    axes[0, digit].imshow(sample, cmap='gray_r', vmin=0, vmax=1)
    axes[0, digit].set_title(str(digit))
    axes[0, digit].axis('off')
    axes[1, digit].imshow(X_images[mask][1], cmap='gray_r', vmin=0, vmax=1)
    axes[1, digit].axis('off')
plt.suptitle('Sample Training Images', fontsize=13)
plt.tight_layout()
plt.savefig('w3_02_samples.png', dpi=100, bbox_inches='tight')
plt.show()

# Convert to PyTorch
X_flat = X_images.reshape(-1, 64)  # Flatten 8x8 → 64
X_tensor = torch.FloatTensor(X_flat).to(device)
y_tensor = torch.LongTensor(y_digits).to(device)

# Split
n_train = int(0.8 * len(X_tensor))
train_data = TensorDataset(X_tensor[:n_train], y_tensor[:n_train])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
X_test = X_tensor[n_train:]
y_test = y_tensor[n_train:]

## Part 2: PyTorch VAE — Clean Implementation

In [ ]:
# ============================================================
# Variational Autoencoder in PyTorch
# ============================================================

class VAE(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=2):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),  # Output in [0, 1]
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """The reparameterization trick: z = mu + std * eps"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """VAE loss = Reconstruction (BCE) + β × KL divergence."""
    recon = F.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


# Create model
vae = VAE(input_dim=64, hidden_dim=128, latent_dim=2).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

print(f"VAE parameters: {sum(p.numel() for p in vae.parameters()):,}")
print(vae)

In [ ]:
# ============================================================
# Training loop
# ============================================================
n_epochs = 100
history = {'total': [], 'recon': [], 'kl': []}

for epoch in range(n_epochs):
    vae.train()
    epoch_loss = 0
    epoch_recon = 0
    epoch_kl = 0
    
    for batch_x, batch_y in train_loader:
        x_hat, mu, logvar = vae(batch_x)
        loss, recon, kl = vae_loss(x_hat, batch_x, mu, logvar, beta=0.5)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_recon += recon.item()
        epoch_kl += kl.item()
    
    n = len(train_loader)
    history['total'].append(epoch_loss / n)
    history['recon'].append(epoch_recon / n)
    history['kl'].append(epoch_kl / n)
    
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d} | Total: {epoch_loss/n:.1f} | "
              f"Recon: {epoch_recon/n:.1f} | KL: {epoch_kl/n:.1f}")

print("\n✓ Training complete!")

In [ ]:
# ============================================================
# Visualize the latent space
# ============================================================
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_tensor)
    mu_np = mu_all.cpu().numpy()
    labels_np = y_tensor.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Latent space colored by digit
scatter = axes[0].scatter(mu_np[:, 0], mu_np[:, 1], c=labels_np, s=10, 
                           cmap='tab10', alpha=0.6)
axes[0].set_title('Latent Space (colored by digit)')
axes[0].set_xlabel('z₁')
axes[0].set_ylabel('z₂')
plt.colorbar(scatter, ax=axes[0], label='Digit')
axes[0].grid(True, alpha=0.2)

# Loss curves
axes[1].plot(history['recon'], label='Reconstruction', alpha=0.8)
axes[1].plot(history['kl'], label='KL Divergence', alpha=0.8)
axes[1].set_title('VAE Loss Components')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('w3_02_latent_space.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 3: GENERATE New Digits!

In [ ]:
# ============================================================
# Generate new images by sampling from latent space
# ============================================================
vae.eval()

# Method 1: Random sampling from N(0,1)
fig, axes = plt.subplots(3, 10, figsize=(15, 5))

for row in range(3):
    for col in range(10):
        z = torch.randn(1, 2).to(device)
        with torch.no_grad():
            generated = vae.decode(z).cpu().numpy().reshape(8, 8)
        axes[row, col].imshow(generated, cmap='gray_r', vmin=0, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('🎨 GENERATED Digits (sampled from N(0,1))', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w3_02_generated.png', dpi=100, bbox_inches='tight')
plt.show()

# Method 2: Interpolation in latent space
print("\nLatent Space Interpolation:")
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for row in range(2):
    z_start = torch.randn(1, 2).to(device)
    z_end = torch.randn(1, 2).to(device)
    
    for col in range(10):
        t = col / 9.0
        z_interp = z_start * (1 - t) + z_end * t
        with torch.no_grad():
            generated = vae.decode(z_interp).cpu().numpy().reshape(8, 8)
        axes[row, col].imshow(generated, cmap='gray_r', vmin=0, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('Latent Space Interpolation (smooth transitions)', fontsize=13)
plt.tight_layout()
plt.savefig('w3_02_interpolation.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ You just GENERATED new images with a neural network!")
print("  This is the foundation of all generative AI.")

In [ ]:
# ============================================================
# Grid walk through latent space
# ============================================================
n_grid = 15
z1 = np.linspace(-3, 3, n_grid)
z2 = np.linspace(-3, 3, n_grid)

canvas = np.zeros((8 * n_grid, 8 * n_grid))

vae.eval()
for i, z1_val in enumerate(z1):
    for j, z2_val in enumerate(z2):
        z = torch.FloatTensor([[z1_val, z2_val]]).to(device)
        with torch.no_grad():
            generated = vae.decode(z).cpu().numpy().reshape(8, 8)
        canvas[j*8:(j+1)*8, i*8:(i+1)*8] = generated

plt.figure(figsize=(10, 10))
plt.imshow(canvas, cmap='gray_r', vmin=0, vmax=1)
plt.title('🗺️ Complete Latent Space Map', fontsize=14, fontweight='bold')
plt.xlabel('z₁')
plt.ylabel('z₂')
plt.savefig('w3_02_latent_grid.png', dpi=100, bbox_inches='tight')
plt.show()

print("This grid shows EVERY point in latent space decoded to an image.")
print("Smooth transitions = the VAE learned a meaningful representation!")

## ✅ Self-Check

- [ ] Your VAE generates recognizable digit-like patterns
- [ ] You can explain the reparameterization trick
- [ ] You see smooth interpolation in latent space
- [ ] You understand: Reconstruction loss learns content, KL divergence organizes the space

## ➡️ Next: `W3_03_PyTorch_Deploy_App.ipynb` — Wrap your model in a web app!

## Visualizing the VAE Architecture

A diagram showing the Probabilistic Encoder, the Reparameterization Trick (sampling `z`), and the Decoder.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
pos = {
    "Input Image": (0, 0),
    "Encoder (Conv)": (2, 0),
    "mu": (4, 1),
    "log_var": (4, -1),
    "Sample (z)": (6, 0),
    "Decoder (ConvT)": (8, 0),
    "Generated Image": (10, 0)
}

G.add_edges_from([
    ("Input Image", "Encoder (Conv)"),
    ("Encoder (Conv)", "mu"),
    ("Encoder (Conv)", "log_var"),
    ("mu", "Sample (z)"),
    ("log_var", "Sample (z)"),
    ("Sample (z)", "Decoder (ConvT)"),
    ("Decoder (ConvT)", "Generated Image")
])

plt.figure(figsize=(12, 4))
nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=3500, node_shape="s", font_size=9)
plt.title("Variational Autoencoder (VAE) Architecture")

plt.savefig('w3_02_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
